In [ ]:
### Esta rotina realizará o gráfico da análise espectral e calculará as frequências mediana, predominante e de 95% da força espectral.

# A rotina deverá ser invocada como:
# forca_total,freqMediana,freqPred,Freq95 = analiseespectral(dados)
# Dados deverá ser o arquivo que possui uma série temporal.
# Criada por Vitor Cimonetti, em 19/04/2026

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import detrend


In [ ]:
df = pd.read_csv('8_cadencia_normal.txt', sep='\t') # sou acostumado com o pandas kk então vou deixar aqui pra não me perder com as colunas
serie1 = np.loadtxt('8_cadencia_normal.txt', delimiter='\t', usecols=(2, 3, 4),skiprows=1)
df.head(10)

In [ ]:
serie1[:,2] = -serie1[:,2] # invertendo o eixo Z pois coloquei o sensor de ponta cabeça

eixoX=serie1[:,0]
eixoY=serie1[:,1]
eixoZ=serie1[:,2]

In [ ]:
fs = 10
t = np.arange(len(serie1)) / fs


plt.figure(figsize=(15, 10))
plt.subplots_adjust(hspace=0.5) # Aqui serve para colocar um espaço maior entre as janelas


plt.subplot(3,1,1)
plt.plot(t,serie1[:,0],color='blue')
plt.ylim(-4,4)
plt.title('Eixo X')
plt.xlabel('Tempo (s)')
plt.ylabel('Amplitude (g)')

plt.subplot(3,1,2)
plt.plot(t,serie1[:,1],color='green')
plt.ylim(-4,4)
plt.title('Eixo Y')
plt.xlabel('Tempo (s)')
plt.ylabel('Amplitude (g)')

plt.subplot(3,1,3)
plt.plot(t,serie1[:,2],color='red')
plt.ylim(-4,4)
plt.title('Eixo Z')
plt.xlabel('Tempo (s)')
plt.ylabel('Amplitude (g)')

In [ ]:
def analiseespectral(dados):
  Freq_aquis=10 # define a frequência de aquisição
  if len(dados) < 4 or not np.isfinite(dados).all():
    raise ValueError('Os dados devem ter pelo menos 4 amostras finitas.')
  tempo=np.arange(len(dados))/Freq_aquis


  plt.figure(figsize=(8,6))
  plt.plot(tempo,dados)
  plt.xlabel('Tempo (s)')
  plt.ylabel('Amplitude do sinal')

  ## Realiza a análise espectral do sinal

  Y=np.fft.fft(detrend(dados,type='constant'),len(dados))  # Calcula os coeficientes, com base em um sinal ajustado pelo detrend
  PY=np.abs(Y)**2/len(dados); # calcula a potência do sinal
  f=Freq_aquis * np.arange(0, len(dados)//2) / len(dados) # define a serie de frequencias


  # Calculo da força espectral e das frequências de interesse.

  forca_total=np.trapezoid(PY[0:len(f)],f)

  # Calcula a frequência mediana

  forca50=forca_total*0.5  # Obtem o valor de força referente a 50% da força total

  i=2
  forca=0
  while forca < forca50 and i <= len(f):
    forca = np.trapezoid(PY[0:i],f[0:i])
    i=i+1

  freqMediana = f[i-2] if forca_total > 0 else np.nan

  # Calcula a frequência de 95% da forca espectral

  forca95=forca_total*0.95; # Obtem o valor de força referente a 95% da força total

  i=2;
  forca=0
  while forca < forca95 and i <= len(f):
    forca = np.trapezoid(PY[0:i],f[0:i]);
    i=i+1

  Freq95 = f[i-2] if forca_total > 0 else np.nan

  # Calcula a frequência predominante

  indice=np.argmax(PY[0:len(dados)//2])
  freqPred=f[indice] if forca_total > 0 else np.nan


  plt.figure(figsize=(8,6))
  plt.plot(f,PY[0:len(f)]) # Gráfico da relação entre potência e frequência
  plt.axvline(x=freqMediana, color='green', linewidth=3, linestyle=':', label = 'Frequência mediana')
  plt.axvline(x=Freq95, color='red', linewidth=3, linestyle=':', label = 'Frequência 95%')
  plt.legend()
  plt.xlabel('Frequencia (Hz)')
  plt.ylabel('Power')



  return forca_total,freqMediana,freqPred,Freq95

In [ ]:
forca_total,freqMediana,freqPred,Freq95 = analiseespectral(eixoZ)

In [ ]:
print('força total: ', forca_total)
print('frequencia mediana: ', freqMediana)
print('frequencia predominante: ', freqPred)
print('frequência de 95% da forca espectral: ', Freq95)